# Phase 5 — Intent Discovery

This notebook discovers and defines the intent taxonomy from the selected brand's customer messages.

**Goal:** Create a small, meaningful set of intents for classification.

**Important:** This notebook works with the actual dataset — no fabricated examples.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.eda_utils import detect_columns
from src.intents.message_extraction import (
    analyze_text_quality,
    identify_customer_messages,
)

INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'
SELECTED_DIR = INTERIM_DIR / 'selected_brand'

## 1. Load Data

In [ ]:
# Load config
config_path = PROJECT_ROOT / 'configs' / 'project.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

selected_brand = config['dataset']['selected_brand']
print(f'Selected brand: {selected_brand}')

# Load discovery messages
discovery_path = SELECTED_DIR / 'intent_discovery_messages.csv'
if discovery_path.exists():
    df = pd.read_csv(discovery_path, low_memory=False)
    print(f'Loaded {len(df):,} customer messages.')
else:
    print('No discovery messages found. Run: python scripts/prepare_intent_discovery.py')
    df = None

In [ ]:
if df is not None:
    col_map = detect_columns(df)
    text_col = col_map.get('text', 'text')
    
    print(f'Columns: {list(df.columns)}')
    print(f'\nSample messages:')
    for i, row in df.head(5).iterrows():
        print(f'  {i+1}. {row[text_col][:100]}')
    
    # Text quality
    quality = analyze_text_quality(df, text_col)
    print(f'\nText quality:')
    for k, v in quality.items():
        print(f'  {k}: {v}')

## 2. TF-IDF Analysis

In [ ]:
if df is not None and len(df) > 0:
    # Use normalized text if available, otherwise raw text
    text_col = 'normalized_text' if 'normalized_text' in df.columns else col_map.get('text', 'text')
    texts = df[text_col].fillna('').astype(str)
    
    # TF-IDF
    tfidf = TfidfVectorizer(
        max_features=5000,
        stop_words='english',
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.8,
    )
    tfidf_matrix = tfidf.fit_transform(texts)
    feature_names = tfidf.get_feature_names_out()
    
    print(f'TF-IDF matrix: {tfidf_matrix.shape}')
    
    # Top terms
    mean_tfidf = tfidf_matrix.mean(axis=0).A1
    top_indices = mean_tfidf.argsort()[-30:][::-1]
    
    print('\nTop 30 terms by mean TF-IDF:')
    for idx in top_indices:
        print(f'  {feature_names[idx]}: {mean_tfidf[idx]:.4f}')

## 3. Clustering for Discovery

In [ ]:
if df is not None and len(df) > 100:
    # Try different K values
    results = []
    K_range = [6, 8, 10, 12, 15]
    
    for k in K_range:
        if k >= len(df):
            continue
        
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(tfidf_matrix)
        
        sil_score = silhouette_score(tfidf_matrix, labels, sample_size=min(5000, len(df)))
        cluster_sizes = pd.Series(labels).value_counts().to_dict()
        
        results.append({
            'k': k,
            'silhouette': round(sil_score, 4),
            'sizes': cluster_sizes,
        })
        print(f'K={k}: silhouette={sil_score:.4f}, sizes={cluster_sizes}')
    
    # Plot silhouette scores
    if results:
        fig, ax = plt.subplots(figsize=(8, 5))
        ks = [r['k'] for r in results]
        sils = [r['silhouette'] for r in results]
        ax.plot(ks, sils, 'bo-')
        ax.set_xlabel('K')
        ax.set_ylabel('Silhouette Score')
        ax.set_title('Clustering Quality vs K')
        plt.tight_layout()
        plt.show()

## 4. Cluster Inspection

In [ ]:
if df is not None and len(df) > 100:
    # Use a reasonable K for inspection
    best_k = results[len(results)//2]['k'] if results else 10
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
    df['cluster'] = kmeans.fit_predict(tfidf_matrix)
    
    # Show representative examples per cluster
    text_col_raw = col_map.get('text', 'text')
    
    print(f'Cluster inspection (K={best_k}):\n')
    for cluster_id in sorted(df['cluster'].unique()):
        cluster_msgs = df[df['cluster'] == cluster_id]
        print(f'\n--- Cluster {cluster_id} ({len(cluster_msgs)} messages) ---')
        
        # Show 5 examples
        for _, row in cluster_msgs.head(5).iterrows():
            text = row[text_col_raw] if text_col_raw in row else str(row)
            print(f'  - {str(text)[:120]}')

## 5. Create Taxonomy

After inspecting clusters, define the intent taxonomy.

**Important:** This must be done manually based on actual data patterns.

In [ ]:
# Taxonomy template - to be filled after cluster inspection
taxonomy = {
    'taxonomy_version': '1.0',
    'selected_brand': selected_brand,
    'created_at': pd.Timestamp.now().isoformat(),
    'intents': [
        # Example structure:
        # {
        #     'intent_id': 'intent_01',
        #     'name': 'order_tracking',
        #     'description': 'Customer asks about order status or delivery',
        #     'in_scope': True,
        #     'examples': ['Where is my order?', 'When will my package arrive?'],
        #     'confusable_with': ['delivery_delay'],
        #     'include': ['Order status inquiries', 'Delivery tracking requests'],
        #     'exclude': ['Refund requests', 'Product complaints']
        # },
    ]
}

# Save template
taxonomy_path = SELECTED_DIR / 'intent_taxonomy.json'
with open(taxonomy_path, 'w') as f:
    json.dump(taxonomy, f, indent=2)
print(f'Taxonomy template saved to: {taxonomy_path}')
print('\nIMPORTANT: Fill in the taxonomy based on cluster inspection.')

## 6. Summary

After completing the taxonomy:

1. Run validation: `python scripts/validate_intent_taxonomy.py`
2. Update `docs/INTENT_LABELING_GUIDE.md`
3. Update `reports/phase_5_intent_analysis.md`